# 07 · Train a custom YOLO detector

**Level:** Intermediate · **Teaching time:** 35–50 minutes plus training
(a CPU can need considerably longer; measure your own rehearsal).

**Prerequisites:** notebooks 01–06; the project Jupyter kernel; internet for the
first pretrained-weight download. A GPU is optional.

**You will:** inspect a labeled dataset, fine-tune a pretrained detector,
read real loss/metric plots, and save a checkpoint for later lessons.

**Demo story:** “A general detector knows everyday objects. Can we teach it our
three custom classes: rectangle, circle, and triangle?” This tiny synthetic
dataset makes the mechanics reproducible. Five epochs demonstrate the workflow;
they do **not** establish high accuracy or real-world readiness.

Run cells from top to bottom. `COURSE_OFFLINE=1` leaves the inspection and
teaching steps available and skips training; `COURSE_SMOKE=1` uses one epoch
on one quarter of the training split. Neither mode fabricates a trained model.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


## Choose the dataset

Leave `CUSTOM_DATA_YAML = None` for the generated shape dataset. To teach your
own classes, set it to a local YAML file; the class names are read from that
file, so the saved model manifest describes the classes you actually trained.
Both `names: [class_a, class_b]` and `names: {0: class_a, 1: class_b}` work.

**Recommended layout:** `images/train`, `images/val`, optional `images/test`,
with matching `labels/train`, `labels/val`, and `labels/test`. Each image has
a same-stem `.txt` label file; create an empty one for a background-only image.
Use `train: images/train`, `val: images/val`, and optionally `test: images/test`
in the YAML. The inspection cells below scan PNG/JPG/JPEG files recursively in
**one directory per split**. Other YOLO layouts such as image-list text files or
multiple directories need adaptations to `split_images` and/or `label_path_for_image`.

An omitted `path` means the YAML's folder. This lesson resolves a relative
`path` against the YAML's folder and creates `artifacts/custom_dataset_resolved.yaml`
with an absolute root for training. This avoids machine-dependent Ultralytics
dataset settings. Your original custom YAML stays unchanged.


In [ ]:
import os, json, time
from datetime import datetime, timezone
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from course_utils import ensure_synthetic_dataset, read_yolo_labels, load_yolo, choose_device

OFFLINE = os.environ.get("COURSE_OFFLINE", "0") == "1"
SMOKE = os.environ.get("COURSE_SMOKE", "0") == "1"
CUSTOM_DATA_YAML = None  # Example: ROOT / "data" / "my_objects" / "data.yaml"
SOURCE_DATA_YAML = ensure_synthetic_dataset() if CUSTOM_DATA_YAML is None else Path(CUSTOM_DATA_YAML).expanduser()
if not SOURCE_DATA_YAML.is_absolute():
    SOURCE_DATA_YAML = ROOT / SOURCE_DATA_YAML
SOURCE_DATA_YAML = SOURCE_DATA_YAML.resolve()
data_spec = yaml.safe_load(SOURCE_DATA_YAML.read_text(encoding="utf-8"))
if not isinstance(data_spec, dict):
    raise ValueError("Dataset YAML must contain a mapping with train, val, and names.")
raw_names = data_spec.get("names")
if isinstance(raw_names, dict):
    names_by_id = {int(key): str(value) for key, value in raw_names.items()}
    if len(names_by_id) != len(raw_names) or sorted(names_by_id) != list(range(len(raw_names))):
        raise ValueError("Class-name IDs must be consecutive integers starting at 0.")
    CLASS_NAMES = [names_by_id[index] for index in range(len(names_by_id))]
elif isinstance(raw_names, list):
    CLASS_NAMES = [str(value) for value in raw_names]
else:
    raise ValueError("Provide names as a list or a zero-based class-ID mapping in the YAML.")
if not CLASS_NAMES or not all(CLASS_NAMES):
    raise ValueError("Provide at least one nonempty class name.")
DATA_DIR = Path(data_spec.get("path") or SOURCE_DATA_YAML.parent).expanduser()
if not DATA_DIR.is_absolute():
    DATA_DIR = SOURCE_DATA_YAML.parent / DATA_DIR
DATA_DIR = DATA_DIR.resolve()
DATA_YAML = SOURCE_DATA_YAML
if CUSTOM_DATA_YAML is not None:
    DATA_YAML = ROOT / "artifacts" / "custom_dataset_resolved.yaml"
    DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
    resolved_spec = {**data_spec, "path": DATA_DIR.as_posix(), "names": dict(enumerate(CLASS_NAMES))}
    DATA_YAML.write_text(yaml.safe_dump(resolved_spec, sort_keys=False), encoding="utf-8")
print("Source YAML:", SOURCE_DATA_YAML)
print("Classes:", CLASS_NAMES)
print(DATA_YAML.read_text(encoding="utf-8"))


## 1. Inspect the training contract before spending compute

Each YOLO label row is `class_id center_x center_y width height`, normalized
by image width/height. Classes start at **0**. The YAML maps each integer to
a name and identifies the train, validation, and test folders.

Train images update weights; validation images help compare settings; hold the
test split aside until the experiment is finished. New photographs of the same
scene can still leak information across splits, even when filenames differ.

**Say, then do:** “Wrong labels produce wrong learning.” Show three label files
and overlay their boxes before pressing Train.


In [ ]:
def split_images(split):
    entry = data_spec.get(split)
    if entry is None and split == "test":
        return []  # A reserved test split is recommended but optional here.
    if not isinstance(entry, str):
        raise ValueError(f"Set {split} to one image directory, or adapt split_images() for your layout.")
    directory = (DATA_DIR / entry).resolve()
    if not directory.is_dir():
        raise ValueError(f"This inspector needs a directory for {split}, not an image-list file: {directory}")
    return sorted(p for p in directory.rglob("*") if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"})

def label_path_for_image(image_path):
    parts = list(image_path.parts)
    if "images" not in parts:
        raise ValueError("Use matching images/... and labels/... folders, or adapt label_path_for_image().")
    parts[max(i for i, part in enumerate(parts) if part == "images")] = "labels"
    return Path(*parts).with_suffix(".txt")

split_paths = {split: split_images(split) for split in ("train", "val", "test")}
if not split_paths["train"] or not split_paths["val"]:
    raise ValueError("Train and validation directories each need at least one PNG/JPG/JPEG image.")
counts = {split: len(paths) for split, paths in split_paths.items()}
display(pd.DataFrame({"split": list(counts), "images": list(counts.values())}))
samples = split_paths["train"][:3]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax in axes:
    ax.axis("off")
for ax, image_path in zip(axes, samples):
    image = Image.open(image_path).convert("RGB")
    label_path = label_path_for_image(image_path)
    boxes, labels = read_yolo_labels(label_path, *image.size)
    if np.any(labels >= len(CLASS_NAMES)):
        raise ValueError(f"Class ID outside YAML names in {label_path}")
    show_boxes(image, boxes, labels, ax=ax, title=image_path.name)
    print(label_path.name, "→", label_path.read_text().strip().splitlines())
plt.tight_layout()
plt.show()


## 2. Make the budget explicit

An **epoch** is a pass over the selected training images. A **batch** determines
how many images contribute to one optimizer step. `imgsz` changes input resolution;
`device` selects the GPU or CPU. Windows notebooks use `workers=0` here so data
loading stays in the notebook process.

**Classroom default:** This small dataset uses CPU training for a predictable
local demonstration. To use an available CUDA GPU, change `DEVICE` below to
`choose_device()`. Rehearse the selected device before presenting; timings depend
on your hardware and dataset.

We use the small YOLO11n checkpoint as a stable teaching baseline. Transfer
learning starts from features learned on a larger dataset and replaces the
class-specific prediction setup for the classes in your YAML.

**Predict:** If the training loss decreases, does that prove generalization?
No: inspect held-out validation behavior too.


In [ ]:
FULL_RUN = False       # Optional: set True for 25 epochs, then rerun from this cell.
EPOCHS = 1 if SMOKE else (25 if FULL_RUN else 5)
TRAIN_FRACTION = 0.25 if SMOKE else 1.0
DEVICE = "cpu"  # Classroom default; change to choose_device() to use an available CUDA GPU.
train_config = dict(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=256, batch=8,
    workers=0, device=DEVICE, seed=42, deterministic=True,
    fraction=TRAIN_FRACTION, amp=False, cache=False,
    project=str(ROOT / "artifacts" / "yolo_runs"),
    name="shapes_yolo11n", exist_ok=False, plots=True,
)
print({k: v for k, v in train_config.items() if k != "data"})
print("Training skipped:", OFFLINE)


## 3. Train and keep an explicit record

This cell performs real optimization and may download pretrained weights on
its first run. Rehearse it before presenting. The console reports individual
loss terms and validation metrics. With a tiny dataset, expect noisy results.

Each run gets its own directory. `best.pt` is chosen using the trainer's
validation criterion; `last.pt` contains the final saved state. The small
manifest points later notebooks to the **actual** saved checkpoint.


In [ ]:
trained_model = None
run_dir = None
manifest_path = ROOT / "artifacts" / "custom_model.json"
if OFFLINE:
    print("Offline mode: training skipped. No new checkpoint or metrics were created.")
else:
    trained_model = load_yolo("yolo11n.pt")
    started = time.perf_counter()
    training_result = trained_model.train(**train_config)
    elapsed = time.perf_counter() - started
    run_dir = Path(trained_model.trainer.save_dir).resolve()
    best_weights = Path(trained_model.trainer.best).resolve()
    if not best_weights.exists():
        raise FileNotFoundError(f"Training did not produce the expected checkpoint: {best_weights}")
    manifest = {
        "best_weights": str(best_weights), "data_yaml": str(DATA_YAML.resolve()),
        "run_dir": str(run_dir), "classes": CLASS_NAMES,
        "epochs_requested": EPOCHS, "train_fraction": TRAIN_FRACTION,
        "seed": 42, "imgsz": 256, "elapsed_seconds": round(elapsed, 2),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "purpose": "short educational run; not a production quality claim",
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Completed in {elapsed / 60:.1f} min. Manifest: {manifest_path}")
    print("Best weights:", best_weights)


## 4. Read the learning curves

A loss is the quantity optimization tries to reduce. Precision, recall, and
mAP evaluate detections and need their own explanation (next lesson).
Training and validation losses need not be numerically identical: augmentation,
sample composition, and evaluation behavior differ.

**Presenter prompt:** Point to an actual epoch on the plot. Ask what evidence
would suggest underfitting or overfitting. One or five epochs rarely support a
confident diagnosis; use a longer, controlled experiment before concluding.


In [ ]:
if run_dir is None:
    print("No training run in this execution. Run the training cell online to plot its log.")
else:
    history = pd.read_csv(run_dir / "results.csv")
    history.columns = history.columns.str.strip()
    display(history.tail())
    loss_cols = [c for c in history if c.startswith(("train/", "val/")) and "loss" in c]
    metric_cols = [c for c in history if c.startswith("metrics/")]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    history.plot(x="epoch", y=loss_cols, ax=axes[0], marker="o", title="Recorded losses")
    if metric_cols:
        history.plot(x="epoch", y=metric_cols, ax=axes[1], marker="o", title="Validation metrics")
    plt.tight_layout()
    plt.show()


## 5. Inspect held-out validation predictions

A low threshold deliberately exposes weak predictions. Empty detections are a
valid result for a short run. Never replace them with ground-truth boxes and
call those model predictions.


In [ ]:
if run_dir is not None:
    from ultralytics import YOLO
    best_model = YOLO(str(best_weights))
    val_images = split_paths["val"][:3]
    predictions = best_model.predict([str(p) for p in val_images], imgsz=256,
                                     conf=0.10, device=DEVICE, verbose=False)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax in axes:
        ax.axis("off")
    for ax, prediction, source in zip(axes, predictions, val_images):
        # Ultralytics plot() returns a BGR image, so reverse channels for Matplotlib.
        ax.imshow(prediction.plot()[..., ::-1])
        ax.set_title(f"{source.name}: {len(prediction.boxes)} predictions")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Prediction display needs this notebook's completed training run.")


## 6. Exercise: run one controlled comparison

1. Keep the split, seed, image size, and starting weights fixed. Compare 5 and
   25 epochs. Record wall time and validation mAP50–95 from each run.
2. Inspect at least three failures. Are they missing objects, wrong classes,
   duplicated boxes, or poor localization?
3. Explain why you should not repeatedly tune settings against the test split.

<details><summary>Solution / discussion</summary>

Change only `FULL_RUN`; rebuild `train_config` and rerun the later cells.
Each training cell creates a fresh model from the same pretrained weights.
Copy each run's path and actual metrics into your experiment notes before
starting another run; the manifest intentionally points to the latest run.
More epochs may improve performance, do nothing, or overfit. Repeated test-set
tuning turns that test set into a validation set and biases your final estimate.
</details>

**Troubleshooting:** CUDA out of memory → reduce `batch` to 4 or 2; keep
`workers=0` on Windows. Slow CPU → use smoke mode for a rehearsal. Download
error → check connectivity and rerun, or use offline mode for the other cells.
Unexpected class names → verify the YAML and zero-based label IDs first.

**Checkpoint:** Explain the split roles, locate `best.pt`, and identify the
manifest. Next: notebook 08 explains what those validation metrics mean.

**Primary references:** [Ultralytics training](https://docs.ultralytics.com/modes/train/),
[detection dataset format](https://docs.ultralytics.com/datasets/detect/),
[YOLO11](https://docs.ultralytics.com/models/yolo11/).
